# 03E. Signal Health Score / GO-NO-GO Engine

Combine existing scoring, decay, regime-conditioned IC, and WFV diagnostics into a diagnostic research health score. This notebook does not create signals, approve trading, change upstream thresholds, or modify scoring, WFV, decay, regime, composite, alpha, stress/freeze, portfolio, or ML logic.


## 1. Purpose and scope

Synthesize existing Phase 2 evidence into one pre-ML signal decision layer. The output is a research ranking and GO-NO-GO gate, not a trading approval layer.


## 2. Imports and config


In [9]:
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.db import get_db_path
from src.run_config import make_run_id, make_run_timestamp
from src.signal_health import (
    build_signal_health_attribution,
    build_signal_health_summary,
    build_signal_health_table,
    load_signal_health_inputs,
)
from src.signal_health_storage import SIGNAL_HEALTH_TABLES, save_signal_health_outputs

HEALTH_VERSION = 'phase2_signal_health_v1'
sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Health version: {HEALTH_VERSION}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Health version: phase2_signal_health_v1


## 3. Create health run_id / timestamp


In [10]:
run_id = make_run_id('phase2_nb03e_signal_health')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: phase2_nb03e_signal_health_20260510_220324
run_timestamp: 2026-05-10 22:03:24


## 4. Load health inputs


In [11]:
health_inputs = load_signal_health_inputs(db_path=sqlite_db_path)

input_shapes = pd.DataFrame(
    [
        {
            'input_name': name,
            'rows': None if df is None else len(df),
            'columns': None if df is None else len(df.columns),
        }
        for name, df in health_inputs.items()
    ]
)

display(input_shapes)


,input_name,rows,columns
0,best_horizon,23,12
1,scoring_gate,92,21
2,decay_summary,60,26
3,regime_opportunity,92,25
4,wfv_gate,15,27


## 5. Build signal health table


In [12]:
signal_health_table = build_signal_health_table(
    best_horizon=health_inputs['best_horizon'],
    scoring_gate=health_inputs['scoring_gate'],
    decay_summary=health_inputs['decay_summary'],
    regime_opportunity=health_inputs['regime_opportunity'],
    wfv_gate=health_inputs.get('wfv_gate'),
    run_id=run_id,
    health_version=HEALTH_VERSION,
)
signal_health_attribution = build_signal_health_attribution(signal_health_table)

health_gate_counts = signal_health_table['signal_health_gate'].value_counts()
top_20_signals = signal_health_table.sort_values(
    ['signal_health_score', 'signal_name', 'horizon'],
    ascending=[False, True, True],
).head(20)
approved_research_signals = signal_health_table.loc[
    signal_health_table['signal_health_gate'].eq('APPROVED_FOR_RESEARCH')
].sort_values(['signal_health_score', 'signal_name', 'horizon'], ascending=[False, True, True])
health_attribution_approved = signal_health_attribution.loc[
    signal_health_attribution['signal_health_gate'].eq('APPROVED_FOR_RESEARCH')
].sort_values(['signal_health_score', 'signal_name', 'horizon'], ascending=[False, True, True])
biggest_penalty_examples = signal_health_attribution.loc[
    signal_health_attribution['total_penalty_points'].lt(0)
].sort_values(['total_penalty_points', 'signal_health_score', 'signal_name'], ascending=[True, True, True]).head(20)

display(signal_health_table.head())
display(signal_health_attribution.head())


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,wfv_status,direction_flip_warning,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,signal_health_score,signal_health_gate,health_notes,run_id,health_version
0,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,...,MISSING_WFV,0,NaN,NaN,NaN,62.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
1,range_expansion_failure_5,20,volatility_structure,POSITIVE_EDGE,WEAK,0.014333,0.014333,0.112310,WATCHLIST,STABLE,...,MISSING_WFV,0,NaN,NaN,NaN,58.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
2,residual_return_vs_universe_20,20,cross_sectional_relative_value,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,-0.013551,0.013551,-0.077233,WATCHLIST,STABLE,...,MISSING_WFV,0,NaN,NaN,NaN,54.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
3,vol_of_vol_20,20,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,...,MISSING_WFV,0,NaN,NaN,NaN,54.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1
4,intraday_reversal_strength_1,1,true_short_term_reversal,POSITIVE_EDGE,WEAK,0.014005,0.014005,0.080810,WATCHLIST,STABLE,...,MISSING_WFV,0,NaN,NaN,NaN,53.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1


,signal_name,horizon,signal_family,signal_health_score,signal_health_gate,ic_strength_points,ic_ir_points,decay_stability_points,sign_stability_points,regime_opportunity_points,wfv_points,direction_flip_penalty,avoid_penalty,no_signal_penalty,low_regime_consistency_penalty,total_positive_points,total_penalty_points,final_score,biggest_positive_driver,biggest_penalty_driver
0,vol_of_vol_20,10,volatility_structure,62.0,WATCHLIST_RESEARCH,10,12,20,8,12,0,0,0,0,0,62,0,62.0,decay_stability_points,NO_PENALTY
1,range_expansion_failure_5,20,volatility_structure,58.0,WATCHLIST_RESEARCH,10,12,20,8,8,0,0,0,0,0,58,0,58.0,decay_stability_points,NO_PENALTY
2,residual_return_vs_universe_20,20,cross_sectional_relative_value,54.0,WATCHLIST_RESEARCH,10,8,20,8,8,0,0,0,0,0,54,0,54.0,decay_stability_points,NO_PENALTY
3,vol_of_vol_20,20,volatility_structure,54.0,WATCHLIST_RESEARCH,10,12,12,8,12,0,0,0,0,0,54,0,54.0,ic_ir_points,NO_PENALTY
4,intraday_reversal_strength_1,1,true_short_term_reversal,53.0,WATCHLIST_RESEARCH,10,8,20,10,5,0,0,0,0,0,53,0,53.0,decay_stability_points,NO_PENALTY


## 6. Build summary


In [13]:
signal_health_summary = build_signal_health_summary(signal_health_table)
display(signal_health_summary)


,n_signals,n_approved,n_watchlist,n_rejected,avg_health_score,max_health_score,best_signal_name,best_signal_horizon,health_version,run_id
0,92,0,9,83,21.336957,62.0,vol_of_vol_20,10,phase2_signal_health_v1,phase2_nb03e_signal_health_20260510_220324


## 7. Save outputs to SQLite


In [14]:
saved_paths = save_signal_health_outputs(
    signal_health_score=signal_health_table,
    signal_health_summary=signal_health_summary,
    signal_health_attribution=signal_health_attribution,
    db_path=sqlite_db_path,
    run_id=run_id,
    health_version=HEALTH_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_HEALTH_TABLES.items()
    ]
)

display(sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,score,signal_health_score_current,signal_health_score_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_health_summary_current,signal_health_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,attribution,signal_health_attribution_current,signal_health_attribution_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 8. Final display


In [15]:
print('Health gate counts')
display(health_gate_counts.rename('signal_horizon_count'))

print('Summary')
display(signal_health_summary)

print('Top signals by health score')
display(top_20_signals)

print('Health attribution for approved signals')
display(health_attribution_approved)

print('Biggest penalty examples')
display(biggest_penalty_examples)

print('SQLite tables written')
display(sqlite_tables_written)


Health gate counts


signal_health_gate
REJECTED_RESEARCH     83
WATCHLIST_RESEARCH     9
Name: signal_horizon_count, dtype: int64

Summary


,n_signals,n_approved,n_watchlist,n_rejected,avg_health_score,max_health_score,best_signal_name,best_signal_horizon,health_version,run_id
0,92,0,9,83,21.336957,62.0,vol_of_vol_20,10,phase2_signal_health_v1,phase2_nb03e_signal_health_20260510_220324


Top signals by health score


...

,signal_name,horizon,signal_family,signal_health_score,signal_health_gate,ic_strength_points,ic_ir_points,decay_stability_points,sign_stability_points,regime_opportunity_points,wfv_points,direction_flip_penalty,avoid_penalty,no_signal_penalty,low_regime_consistency_penalty,total_positive_points,total_penalty_points,final_score,biggest_positive_driver,biggest_penalty_driver


Biggest penalty examples


,signal_name,horizon,signal_family,signal_health_score,signal_health_gate,ic_strength_points,ic_ir_points,decay_stability_points,sign_stability_points,regime_opportunity_points,wfv_points,direction_flip_penalty,avoid_penalty,no_signal_penalty,low_regime_consistency_penalty,total_positive_points,total_penalty_points,final_score,biggest_positive_driver,biggest_penalty_driver
90,vol_surprise_20_60,1,volatility_structure,0.0,REJECTED_RESEARCH,0,8,0,0,0,0,0,-20,-10,-10,8,-40,0.0,ic_ir_points,avoid_penalty
80,dollar_volume_shock_20,1,liquidity_flow,0.0,REJECTED_RESEARCH,0,0,20,8,0,0,0,-20,-10,0,28,-30,0.0,decay_stability_points,avoid_penalty
81,dollar_volume_shock_20,20,liquidity_flow,0.0,REJECTED_RESEARCH,0,0,12,0,0,0,0,-20,-10,0,12,-30,0.0,decay_stability_points,avoid_penalty
86,overnight_gap_reversal_1,1,true_short_term_reversal,0.0,REJECTED_RESEARCH,0,0,12,5,0,0,0,-20,-10,0,17,-30,0.0,decay_stability_points,avoid_penalty
87,overnight_gap_reversal_1,5,true_short_term_reversal,0.0,REJECTED_RESEARCH,0,0,12,0,5,0,0,0,-10,-10,17,-20,0.0,decay_stability_points,no_signal_penalty
88,three_day_overextension_reversal,10,true_short_term_reversal,0.0,REJECTED_RESEARCH,0,0,12,0,8,0,0,0,-10,-10,20,-20,0.0,decay_stability_points,no_signal_penalty
89,three_day_overextension_reversal,20,true_short_term_reversal,0.0,REJECTED_RESEARCH,0,0,12,0,5,0,0,0,-10,-10,17,-20,0.0,decay_stability_points,no_signal_penalty
91,vol_surprise_20_60,20,volatility_structure,0.0,REJECTED_RESEARCH,0,8,0,0,5,0,0,0,-10,-10,13,-20,0.0,ic_ir_points,no_signal_penalty
76,relative_return_zscore_60,20,cross_sectional_relative_value,4.0,REJECTED_RESEARCH,0,0,12,0,12,0,0,0,-10,-10,24,-20,4.0,decay_stability_points,no_signal_penalty
75,liquidity_adjusted_reversal_5,5,liquidity_flow,5.0,REJECTED_RESEARCH,0,0,12,5,8,0,0,0,-10,-10,25,-20,5.0,decay_stability_points,no_signal_penalty


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,score,signal_health_score_current,signal_health_score_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_health_summary_current,signal_health_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,attribution,signal_health_attribution_current,signal_health_attribution_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [16]:
from src.db import load_table

health = load_table("signal_health_score_current")

print("Shape:", health.shape)
print("\nGate counts:")
print(health["signal_health_gate"].value_counts())

print("\nScore range:")
print(health["signal_health_score"].describe())

print("\nApproved signals:")
display(
    health[health["signal_health_gate"] == "APPROVED_FOR_RESEARCH"]
    [["signal_name", "horizon", "signal_family", "signal_health_score",
      "best_abs_mean_ic", "decay_risk_flag", "adjusted_best_abs_ic",
      "recommended_use", "wfv_status"]]
    .sort_values("signal_health_score", ascending=False)
)

print("\nAny invalid scores?")
print(((health["signal_health_score"] < 0) | (health["signal_health_score"] > 100)).any())

print("\nAny APPROVED with AVOID / HIGH_DECAY / direction flip?")
display(
    health[
        (health["signal_health_gate"] == "APPROVED_FOR_RESEARCH") &
        (
            (health["recommended_use"] == "AVOID") |
            (health["decay_risk_flag"] == "HIGH_DECAY_RISK") |
            (health["direction_flip_warning"] == 1)
        )
    ]
)

Shape: (92, 31)

Gate counts:
signal_health_gate
REJECTED_RESEARCH     83
WATCHLIST_RESEARCH     9
Name: count, dtype: int64

Score range:
count    92.000000
mean     21.336957
std      16.350742
min       0.000000
25%       8.000000
50%      16.000000
75%      33.000000
max      62.000000
Name: signal_health_score, dtype: float64

Approved signals:


,signal_name,horizon,signal_family,signal_health_score,best_abs_mean_ic,decay_risk_flag,adjusted_best_abs_ic,recommended_use,wfv_status



Any invalid scores?
False

Any APPROVED with AVOID / HIGH_DECAY / direction flip?


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,wfv_status,direction_flip_warning,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,signal_health_score,signal_health_gate,health_notes,run_id,health_version
